# Lab 6: Covert Channels in Image Metadata
**Guided investigation | Student notebook**

**Scenario:** Six images contain pieces of a fictional cybersecurity incident narrative. Each image contains **one partial secret**. Your task is to find the six pieces, document how each was recovered, and reconstruct the full message in the correct order. **Do not execute any activity described by the messages.**

**Learning goals:** 
-  distinguish EXIF, PNG text, and XMP metadata; 
-  use short Python snippets to extract metadata;
-  recognize encoded text; 
-  record reproducible forensic evidence; 
-  assess the limitations of metadata-based conclusions.

**How this lab works:** First practise each small snippet on **one image**. Then select and run the relevant snippets on **each of the six images**, one at a time. You will write the final secret yourself. **No completed secret is provided in this student notebook.**

## 0. Prepare the environment
Keep this notebook beside the `Lab6` folder, which contains `Images`. Run the following two short cells.

In [1]:
%pip -q install Pillow

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
from PIL import Image, ExifTags

image_dir = Path("Images")
print("Images found:", len(list(image_dir.glob("*.png"))))

Images found: 6


### Your evidence log
For **each image**, record the metadata type, field name, raw value (or a short excerpt), decoding method (if any), recovered message, and your interpretation. Do not assume that `getexif()` displays all metadata.

## 1. First technique: EXIF metadata (one image)
EXIF metadata can include camera details, dates, descriptions, and other tags. Try **Brain-Channel.png** first.

In [12]:
image_path = image_dir / "Music.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    print("EXIF tags:", len(exif))
    for tag_id, value in exif.items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)

EXIF tags: 9
ResolutionUnit : 2
ExifOffset : 264
Make : Olympus
Model : OM-D E-M10
DateTime : 2025:02:06 11:11:00
YCbCrPositioning : 1
Copyright : RVhGSUw6IFR1bm5lbCBjb21wcmVzc2VkIGRhdGFiYXNlIGR1bXAgdmlhIEROUyBDb3ZlcnQgQ2hhbm5lbC4=
XResolution : 72.0
YResolution : 72.0


**Observe:** Which EXIF field carries a message? Record it in your evidence log.

**Try it yourself:** Change `image_path` to a different image. If `EXIF tags: 0` appears, does that mean the image has no metadata? Explain.

In [ ]:
# Your turn: choose another image and repeat the EXIF check.
image_path = image_dir / "Activation.png"
# TODO: reuse the short EXIF snippet above

## 2. Second technique: PNG text metadata (one image)
PNG files can contain textual information that is **not** returned by `getexif()`. Try **Brain.png**.

In [6]:
image_path = image_dir / "Chemicals.png"
with Image.open(image_path) as img:
    for key, value in img.info.items():
        if isinstance(value, str):
            print(key, ":", value)

ImageDescription : EXPLOIT: Deploy SQL Injection payload to bypass WAF.


**Observe:** Which PNG text field contains a clue? Try **Chemicals.png** next using the same short snippet.

In [ ]:
image_path = image_dir / "Chemicals.png"
# TODO: reuse the PNG text snippet above

## 3. Third technique: XMP metadata (one image)
XMP is XML-based metadata. It may be stored in PNG files even when EXIF is empty. Try **Dopamine.png**.

In [7]:
image_path = image_dir / "Dopamine.png"
with Image.open(image_path) as img:
    xmp = img.info.get("XML:com.adobe.xmp") or img.info.get("xmp")
    if isinstance(xmp, bytes):
        xmp = xmp.decode("utf-8", errors="replace")
    print(xmp if xmp else "No XMP found")

<?xpacket begin='﻿' id='W5M0MpCehiHzreSzNTczkc9d'?>
<x:xmpmeta xmlns:x='adobe:ns:meta/' x:xmptk='Image::ExifTool 12.76'>
<rdf:RDF xmlns:rdf='http://www.w3.org/1999/02/22-rdf-syntax-ns#'>

 <rdf:Description rdf:about=''
  xmlns:exif='http://ns.adobe.com/exif/1.0/'>
  <exif:MakerNote>PERSIST: Establish reverse shell via obfuscated cron job.</exif:MakerNote>
 </rdf:Description>
</rdf:RDF>
</x:xmpmeta>
<?xpacket end='r'?>


**Observe:** Locate the XML element containing the partial secret. Why did the EXIF-only approach miss it? **Note:** XMP may appear under either `XML:com.adobe.xmp` or `xmp`, depending on how the file was created and the software version.

## 4. Fourth technique: Decode suspicious text (one image)
Sometimes metadata contains **encoded** rather than readable text. Try **Music.png**. First inspect its EXIF or PNG text fields. Identify a value that resembles Base64, then decode it using this small example.

In [13]:
import base64

example = "RVhGSUw6IFR1bm5lbCBjb21wcmVzc2VkIGRhdGFiYXNlIGR1bXAgdmlhIEROUyBDb3ZlcnQgQ2hhbm5lbC4="
print(base64.b64decode(example).decode("utf-8"))

EXFIL: Tunnel compressed database dump via DNS Covert Channel.


**Your turn:** Replace `example` with the suspicious metadata value from **Music.png**. Do not assume every long string is Base64; use the context and validate the result.

In [ ]:
encoded_message = ""  # TODO: paste the encoded value found in Music.png
if encoded_message:
    print(base64.b64decode(encoded_message).decode("utf-8"))

## 5. Your investigation: six images, one by one
Use **only the relevant short snippets** from Sections 1–4 for each image. For each image: (1) try EXIF; (2) if the clue is not visible, inspect PNG text; (3) inspect XMP if present; (4) decode only when needed; (5) document the partial secret.

| Image | Metadata type / field | Decoding needed? | Partial secret recovered |
|---|---|---|---|
| Activation.png | | | |
| Brain-Channel.png | | | |
| Brain.png | | | |
| Chemicals.png | | | |
| Dopamine.png | | | |
| Music.png | | | |

**Important:** The filename order is **not** necessarily the narrative order.

### Investigation 1: `Activation.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [ ]:
image_path = image_dir / "Activation.png"
# TODO: paste or adapt the relevant short snippet from Sections 1–4

In [16]:
image_path = image_dir / "Activation.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    print("=== Exif sub-IFD ===")
    try:
        exif_ifd = exif.get_ifd(0x8769)
        for tag_id, value in exif_ifd.items():
            print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)
    except Exception as e:
        print("No Exif sub-IFD:", e)
    print("=== GPS IFD ===")
    try:
        gps_ifd = exif.get_ifd(0x8825)
        for tag_id, value in gps_ifd.items():
            print(ExifTags.GPSTAGS.get(tag_id, tag_id), ":", value)
    except Exception as e:
        print("No GPS IFD:", e)



=== Exif sub-IFD ===
ExifVersion : b'0232'
ComponentsConfiguration : b'\x01\x02\x03\x00'
FlashPixVersion : b'0100'
DateTimeOriginal : 2025:02:02 10:15:00
DateTimeDigitized : 2025:02:02 10:15:00
ColorSpace : 65535
UserComment : b'ASCII\x00\x00\x00VULN: Identify unpatched CVE-2024-XXXX in web gateway.'
=== GPS IFD ===
GPSVersionID : b'\x02\x03\x00\x00'
GPSLatitudeRef : N
GPSLatitude : (40.0, 42.0, 46.08)
GPSLongitudeRef : W
GPSLongitude : (74.0, 0.0, 21.6)


### Investigation 2: `Brain-Channel.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [ ]:
image_path = image_dir / "Brain-Channel.png"
# TODO: paste or adapt the relevant short snippet from Sections 1–4

### Investigation 3: `Brain.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [ ]:
image_path = image_dir / "Brain.png"
# TODO: paste or adapt the relevant short snippet from Sections 1–4

### Investigation 4: `Chemicals.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [ ]:
image_path = image_dir / "Chemicals.png"
# TODO: paste or adapt the relevant short snippet from Sections 1–4

### Investigation 5: `Dopamine.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [ ]:
image_path = image_dir / "Dopamine.png"
# TODO: paste or adapt the relevant short snippet from Sections 1–4

### Investigation 6: `Music.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [ ]:
image_path = image_dir / "Music.png"
# TODO: paste or adapt the relevant short snippet from Sections 1–4

## 6. Reconstruct the complete secret
Arrange the six recovered fragments into a logical sequence. Use the **stage labels** found in the evidence to justify your order. Do **not** use image filenames as the ordering rule.

1. **Stage:** __________  **Recovered message:** __________________________________
2. **Stage:** __________  **Recovered message:** __________________________________
3. **Stage:** __________  **Recovered message:** __________________________________
4. **Stage:** __________  **Recovered message:** __________________________________
5. **Stage:** __________  **Recovered message:** __________________________________
6. **Stage:** __________  **Recovered message:** __________________________________

**Explain your reasoning:** Why is this the most plausible order?

## 7. Build an automated metadata scanner
**Learning goal:** Combine the short snippets from Parts 1–4 into a reusable tool, as you did when developing security scanners in Lab 5.

Your scanner must:
1. Loop through **all six images** automatically.
2. Extract **EXIF**, **PNG text**, and **XMP** metadata where available.
3. Record each image's filename, format, metadata field names, and any suspected hidden-message field(s).
4. Identify encoded content and explain any decoding method used; do not automatically assume all unusual text is malicious.
5. Assign a **simple, explainable risk score** and record the reason for each score.
6. Export one row per image to `metadata_report.csv`.

**Suggested classroom risk model:** +10 for a confirmed covert-message field, +5 for a suspicious encoded value requiring investigation, and +5 for an additional unusual metadata indicator supported by evidence. Avoid double-counting an encoded message as both an encoded indicator and an additional unusual indicator. These scores are **heuristics**, not forensic probabilities.

**Step 7A — Starter: loop through the six images.**

In [17]:
for image_path in sorted(image_dir.glob("*.png")):
    with Image.open(image_path) as img:
        print(image_path.name, "→", img.format, "| fields:", len(img.info))


Activation.png → PNG | fields: 8
Brain-Channel.png → PNG | fields: 4
Brain.png → PNG | fields: 4
Chemicals.png → PNG | fields: 1
Dopamine.png → PNG | fields: 5
Music.png → PNG | fields: 8


**Step 7B — Starter: create one result row.** Build on this structure rather than copying a complete solution.

In [18]:
image_path = image_dir / "Brain.png"
with Image.open(image_path) as img:
    row = {
        "filename": image_path.name,
        "format": img.format,
        "metadata_fields": ", ".join(img.info.keys()),
        "risk_score": 0,
    }
print(row)

{'filename': 'Brain.png', 'format': 'PNG', 'metadata_fields': 'srgb, gamma, dpi, Software', 'risk_score': 0}


**Step 7C — Your implementation:** Complete the scanner below or create `metadata_scanner.py`.

Reuse the snippets you have already practised. Extend the result dictionary to include `suspected_field`, `encoding`, and `reason`. The scanner should record the relevant evidence and explain why a risk score was assigned. You may use `csv.DictWriter` from the starter code. **Do not hard-code the six secret messages as the detector's output.**

In [ ]:
import csv

results = []

for image_path in sorted(image_dir.glob("*.png")):
    with Image.open(image_path) as img:
        result = {
            "filename": image_path.name,
            "format": img.format,
            "metadata_fields": ", ".join(img.info.keys()),
            "suspected_field": "",    # TODO: identify relevant EXIF, PNG, or XMP field
            "encoding": "",           # TODO: indicate Base64, plain text, etc.
            "risk_score": 0,          # TODO: apply and justify your scoring rules
            "reason": "",             # TODO: explain evidence for this score
        }
        # TODO: inspect EXIF metadata
        # TODO: inspect PNG text metadata and XMP metadata
        # TODO: identify covert-message indicators and update result
        results.append(result)

with open("metadata_report.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print("Saved", len(results), "rows to metadata_report.csv")

In [19]:
import base64
import csv
from pathlib import Path

from PIL import ExifTags, Image

image_dir = Path("Images")

MESSAGE_FIELDS = {
    "EXIF": ("ImageDescription", "UserComment", "Copyright"),
    "EXIF sub-IFD": ("ImageDescription", "UserComment", "Copyright"),
    "PNG text": ("Software", "ImageDescription"),
    "XMP": ("MakerNote",),
}

def text_of(value):

    if isinstance(value, bytes):

        if value[:8] in (b"ASCII\x00\x00\x00", b"UNICODE\x00", b"JIS\x00\x00\x00\x00"):
            value = value[8:]
        return value.decode("utf-8", errors="replace")
    return str(value)

def try_base64(value):

    try:
        decoded = base64.b64decode(value, validate=True).decode("utf-8")
    except Exception:
        return None
    if decoded.strip() and decoded.isprintable():
        return decoded
    return None

def collect_fields(img):

    collected = []

    exif = img.getexif()
    for tag_id, value in exif.items():
        name = ExifTags.TAGS.get(tag_id, tag_id)
        collected.append(("EXIF", str(name), text_of(value)))

    try:
        sub_ifd = exif.get_ifd(0x8769)
    except Exception:
        sub_ifd = {}
    for tag_id, value in sub_ifd.items():
        name = ExifTags.TAGS.get(tag_id, tag_id)
        collected.append(("EXIF sub-IFD", str(name), text_of(value)))

    for key, value in img.info.items():
        if isinstance(value, str):
            collected.append(("PNG text", key, value))

    xmp = img.info.get("XML:com.adobe.xmp") or img.info.get("xmp")
    if isinstance(xmp, bytes):
        xmp = xmp.decode("utf-8", errors="replace")
    if isinstance(xmp, str) and xmp:
        collected.append(("XMP", "XMP packet", xmp))

        start_tag = "<exif:MakerNote>"
        end_tag = "</exif:MakerNote>"
        start = xmp.find(start_tag)
        end = xmp.find(end_tag)
        if start != -1 and end != -1 and end > start:
            maker_note = xmp[start + len(start_tag):end]
            collected.append(("XMP", "MakerNote", maker_note))

    return collected

results = []

for image_path in sorted(image_dir.glob("*.png")):

    try:
        with Image.open(image_path) as img:
            image_format = img.format
            fields = collect_fields(img)
    except Exception as error:
        print(image_path.name, "could not be opened:", error)
        results.append({
            "filename": image_path.name,
            "format": "",
            "metadata_fields": "",
            "suspected_field": "",
            "encoding": "",
            "risk_score": 0,
            "reason": "File could not be opened, so no metadata was checked.",
        })
        continue

    field_names = []
    for container, name, value in fields:
        if name not in field_names and name != "XMP packet":
            field_names.append(name)

    scored = set()
    risk_score = 0
    suspected = []
    encoding = ""
    reasons = []

    candidates = []
    seen_values = set()
    for container, name, value in fields:

        if (name, value) in seen_values:
            continue
        seen_values.add((name, value))
        if name in MESSAGE_FIELDS.get(container, ()) and value.strip():
            candidates.append((container, name, value))

    if candidates:
        container, name, value = candidates[0]
        suspected.append(container + " / " + name)
        if "covert-message field" not in scored:
            scored.add("covert-message field")
            risk_score += 10
            reasons.append(
                "Suspected covert-message field found in " + container
                + " field " + name + ", giving 10 points."
            )

        if try_base64(value) is not None:
            encoding = "Base64"
            if "encoded value" not in scored:
                scored.add("encoded value")
                risk_score += 5
                reasons.append(
                    "Suspected value decodes from Base64 to readable "
                    "text, giving 5 points for an encoded value."
                )
        else:
            encoding = "plain text"

        if len(candidates) > 1:
            second_container, second_name, second_value = candidates[1]
            suspected.append(second_container + " / " + second_name)
            if "additional indicator" not in scored:
                scored.add("additional indicator")
                risk_score += 5
                reasons.append(
                    "Second suspected field found in " + second_container
                    + " field " + second_name
                    + ", giving 5 points for an additional indicator."
                )
    else:
        reasons.append("No suspected message field found.")

    result = {
        "filename": image_path.name,
        "format": image_format,
        "metadata_fields": ", ".join(field_names),
        "suspected_field": "; ".join(suspected),
        "encoding": encoding,
        "risk_score": risk_score,
        "reason": " ".join(reasons),
    }
    results.append(result)

    print(
        result["filename"], "|", result["format"],
        "| suspected:", result["suspected_field"] or "none",
        "| encoding:", result["encoding"] or "none",
        "| score:", result["risk_score"],
    )

with open("metadata_report.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print("Saved", len(results), "rows to metadata_report.csv")


Activation.png | PNG | suspected: EXIF sub-IFD / UserComment | encoding: plain text | score: 10
Brain-Channel.png | PNG | suspected: EXIF / ImageDescription | encoding: plain text | score: 10
Brain.png | PNG | suspected: PNG text / Software | encoding: plain text | score: 10
Chemicals.png | PNG | suspected: PNG text / ImageDescription | encoding: plain text | score: 10
Dopamine.png | PNG | suspected: XMP / MakerNote | encoding: plain text | score: 10
Music.png | PNG | suspected: EXIF / Copyright | encoding: Base64 | score: 15
Saved 6 rows to metadata_report.csv


**Reflection questions**

1. Why can an image contain a hidden message even when `getexif()` returns no tags?
2. How is **encoding** different from **encryption**? Is Base64 a secure way to hide information?
3. Why should investigators preserve the original files and record hashes?
4. Can metadata alone prove that an image was edited or that an incident occurred? What corroborating evidence would you need?
5. How does this investigation connect to the automated evidence collection and analysis you practised in Labs 4 and 5?



## Submission requirements
Submit **one PDF** named `Lab6-FirstName-LastName-StudentNumber.pdf` that includes:

- **Metadata evidence table** for all six images, identifying the metadata type, field, relevant raw value, decoding method (if applicable), and recovered fragment.
- **Screenshots of the extraction output** showing how each partial secret was found.
- **Complete reconstructed six-stage secret** in the correct order, with a brief explanation of the sequence.
- **Your completed Python metadata scanner** (code and brief explanation of its main decisions).
- **Scanner results:** `metadata_report.csv` output as a table, including the risk score and supporting reason for each image.
- **Brief interpretation:** Explain one limitation of EXIF-only extraction and one limitation of automated risk scoring.
- **Answer to reflection questions** 

Do not execute the activities described by the recovered messages.

## Grading rubric (100 points)

| Component | Points | Full-credit expectations |
|---|---:|---|
| Metadata extraction and evidence documentation | **25** | Appropriate EXIF/PNG/XMP extraction for all six images; accurate field identification; clear evidence screenshots and tables. |
| Covert-channel detection and decoding | **25** | All six hidden fragments recovered accurately; encoded content decoded correctly; extraction methods explained. |
| Six-stage secret reconstruction | **20** | Complete secret assembled in correct order with a logical justification. |
| Automated metadata scanner and CSV report | **20** | Scanner iterates through all six images, checks relevant metadata types, identifies suspected fields, exports complete CSV rows, and applies consistent, explainable risk scores. |
| Interpretations, reflections, code quality, and presentation | **10** | Readable, reproducible Python code; clear report organization; thoughtful explanation of EXIF-only and risk-scoring limitations. |
| **Total** | **100** | |